### Mounting Google Drive

Mounting your Google Drive allows Colab to access files stored there directly. This is a robust solution for working with folders and persistent data.

Run the following code cell to mount your Google Drive. You will be prompted to authenticate your Google account.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Once your Drive is mounted, your files and folders will be accessible under `/content/drive/MyDrive/`. For example, if you have a folder named `my_data` in your Google Drive, you can access it at `/content/drive/MyDrive/my_data`.

In [ ]:
# Example: List contents of your Google Drive's root folder
import os

# Change this path to point to your specific folder(s) on Google Drive
# For example, if you have a folder named 'my_data' in your Drive, change it to '/content/drive/MyDrive/my_data'
drive_folder_path = '/content/drive/MyDrive/Colab Notebooks/train'

if os.path.exists(drive_folder_path):
    print(f"Contents of '{drive_folder_path}':")
    for item in os.listdir(drive_folder_path):
        print(item)
else:
    print(f"The path '{drive_folder_path}' does not exist. Please ensure your Drive is mounted and the path is correct.")

In [25]:
import os
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class ClassificationDataset(Dataset):
    def __init__(
        self,
        root_dir,
        class_to_idx=None,
        image_size=(224, 224),
        transform=None,
    ):
        """
        Args:
            root_dir: Path to train/ or validation/
            class_to_idx: Existing class mapping. If None, create it
                          from the classes found in root_dir.
            image_size: (height, width) used to resize images.
            transform: Optional additional torchvision transforms.
        """

        self.root_dir = root_dir

        # Find class folders
        class_names = sorted(
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        )

        # If no mapping is provided, create it.
        # This is normally done for the training set.
        if class_to_idx is None:
            self.class_to_idx = {
                class_name: idx
                for idx, class_name in enumerate(class_names)
            }
        else:
            # Copy it so we don't modify the original mapping
            self.class_to_idx = dict(class_to_idx)

            # Add classes that are only present in this dataset
            next_idx = max(self.class_to_idx.values(), default=-1) + 1

            for class_name in class_names:
                if class_name not in self.class_to_idx:
                    self.class_to_idx[class_name] = next_idx
                    next_idx += 1

        # Default transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize(image_size),
                transforms.ToTensor(),
            ])
        else:
            self.transform = transform

        # Collect all (image_path, class_id) pairs
        self.samples = []

        image_extensions = {
            ".jpg", ".jpeg", ".png",
            ".bmp", ".webp", ".tif", ".tiff"
        }

        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            class_id = self.class_to_idx[class_name]

            for filename in os.listdir(class_dir):
                image_path = os.path.join(class_dir, filename)

                if not os.path.isfile(image_path):
                    continue

                extension = os.path.splitext(filename)[1].lower()

                if extension in image_extensions:
                    self.samples.append(
                        (image_path, class_id)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, class_id = self.samples[index]

        image = Image.open(image_path).convert("RGB")

        image = self.transform(image)

        return image, class_id

In [86]:
train_dataset = ClassificationDataset(
    "/content/drive/MyDrive/Colab Notebooks/train",
    image_size=(224, 224)
)

val_dataset = ClassificationDataset(
    "/content/drive/MyDrive/Colab Notebooks/validation",
    class_to_idx=train_dataset.class_to_idx,
    image_size=(224, 224)
)

In [28]:
print("Train mapping:")
print(train_dataset.class_to_idx)

print("\nValidation mapping:")
print(val_dataset.class_to_idx)

Train mapping:
{'10043': 0, '10053': 1, '10061': 2, '10062': 3, '10075': 4, '10086': 5, '10119': 6, '10147': 7, '10159': 8, '10160': 9, '10190': 10, '10226': 11, '10258': 12, '10325': 13, '10377': 14, '10408': 15, '10444': 16, '10445': 17, '10465': 18, '10470': 19, '10472': 20, '10489': 21, '10492': 22, '10495': 23, '10518': 24, '10519': 25, '12390': 26, '12406': 27, '12407': 28, '12408': 29, '12409': 30, '12410': 31, '12411': 32, '12412': 33, '12413': 34, '12414': 35, '12415': 36, '12416': 37, '12417': 38, '12418': 39, '12419': 40, '12420': 41, '12421': 42, '12425': 43, '12427': 44, '12675': 45, '12676': 46, '12677': 47, '12681': 48, '12683': 49, '12684': 50, '12685': 51, '12687': 52, '12688': 53, '12689': 54, '12690': 55, '12693': 56, '1293': 57, '251': 58, '541': 59}

Validation mapping:
{'10043': 0, '10053': 1, '10061': 2, '10062': 3, '10075': 4, '10086': 5, '10119': 6, '10147': 7, '10159': 8, '10160': 9, '10190': 10, '10226': 11, '10258': 12, '10325': 13, '10377': 14, '10408': 15,

In [30]:
def check_mappings(train_dataset, val_dataset):

    train_mapping = train_dataset.class_to_idx
    val_mapping = val_dataset.class_to_idx

    # 1. Every training class must exist in validation
    missing_from_val = set(train_mapping) - set(val_mapping)

    # 2. Every shared class must have exactly the same ID
    mismatches = {}

    for class_name, train_id in train_mapping.items():
        val_id = val_mapping[class_name]

        if train_id != val_id:
            mismatches[class_name] = (train_id, val_id)

    if mismatches:
        print("ERROR: Class ID mismatches:")
        for class_name, (train_id, val_id) in mismatches.items():
            print(
                f"  {class_name}: "
                f"train={train_id}, validation={val_id}"
            )
        return False

    # 3. Find classes that exist only in validation
    extra_classes = set(val_mapping) - set(train_mapping)

    print("Mapping is consistent.")

    if extra_classes:
        print("Additional validation classes:")
        for class_name in sorted(extra_classes):
            print(
                f"  {class_name} -> {val_mapping[class_name]}"
            )
    else:
        print("No additional validation classes.")

    return True

check_mappings(train_dataset, val_dataset)

Mapping is consistent.
Additional validation classes:
  502 -> 60


True

In [31]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=4
)

for images, labels in train_loader:
    print("Images:", images.shape)
    print("Labels:", labels.shape)
    print("Labels:", labels)

    break

Images: torch.Size([2, 3, 224, 224])
Labels: torch.Size([2])
Labels: tensor([42, 40])


In [67]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import ResNet18_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 1e-4

weights = ResNet18_Weights.IMAGENET1K_V1

model = models.resnet18(weights=weights)

train_classes = sorted(train_dataset.class_to_idx.values())

# Replace ImageNet classifier
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(train_classes)
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR
)

In [55]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import ResNet18_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 1e-4


model = models.resnet18()

train_classes = sorted(train_dataset.class_to_idx.values())

# Replace ImageNet classifier
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(train_classes)
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR
)

In [69]:
class_to_model_idx = {
    class_id: idx
    for idx, class_id in enumerate(train_classes)
}

best_accuracy = 0.0
best_model_path = "/content/drive/MyDrive/Colab Notebooks/resnet18_finetuned.pth"

for epoch in range(15):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # Convert actual class IDs to model indices
        labels = torch.tensor(
            [class_to_model_idx[label.item()] for label in labels],
            device=DEVICE
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    print(
        f"Epoch [{epoch+1}/{15}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f}"
    )

    if train_accuracy >= best_accuracy:

        best_accuracy = train_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print(
            f"Saved new best model "
            f"(accuracy={best_accuracy:.4f})"
        )

    else:

        print(
            f"Accuracy dropped "
            f"({train_accuracy:.4f} < {best_accuracy:.4f}). "
            f"Stopping training."
        )

        break


Epoch [1/15] Train Loss: 2.0878 Train Acc: 0.5954
Saved new best model (accuracy=0.5954)
Epoch [2/15] Train Loss: 1.1271 Train Acc: 0.7756
Saved new best model (accuracy=0.7756)
Epoch [3/15] Train Loss: 0.7986 Train Acc: 0.8558
Saved new best model (accuracy=0.8558)
Epoch [4/15] Train Loss: 0.5525 Train Acc: 0.9215
Saved new best model (accuracy=0.9215)
Epoch [5/15] Train Loss: 0.4857 Train Acc: 0.9271
Saved new best model (accuracy=0.9271)
Epoch [6/15] Train Loss: 0.3669 Train Acc: 0.9399
Saved new best model (accuracy=0.9399)
Epoch [7/15] Train Loss: 0.3244 Train Acc: 0.9511
Saved new best model (accuracy=0.9511)
Epoch [8/15] Train Loss: 0.2665 Train Acc: 0.9591
Saved new best model (accuracy=0.9591)
Epoch [9/15] Train Loss: 0.1794 Train Acc: 0.9728
Saved new best model (accuracy=0.9728)
Epoch [10/15] Train Loss: 0.2047 Train Acc: 0.9647
Accuracy dropped (0.9647 < 0.9728). Stopping training.


In [72]:
import torch
import torch.nn.functional as F

THRESHOLD = 0.6

model.eval()

# Classes seen during training
train_classes_tensor = torch.tensor(
    train_classes,
    device=DEVICE
)

# Per-class statistics
class_correct = {
    class_id: 0
    for class_id in train_classes
}

class_total = {
    class_id: 0
    for class_id in train_classes
}

# Overall known statistics
known_correct = 0
known_total = 0

# Unknown statistics
unknown_correct = 0
unknown_total = 0


with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # ---------------------------------------------
        # Forward pass
        # ---------------------------------------------

        outputs = model(images)

        probs = F.softmax(outputs, dim=1)

        max_probs, pred_indices = torch.max(
            probs,
            dim=1
        )

        # Convert model indices -> actual class IDs
        pred_classes = torch.tensor(
            [
                train_classes[i]
                for i in pred_indices.cpu().tolist()
            ],
            device=DEVICE
        )

        # ---------------------------------------------
        # Abstention
        # ---------------------------------------------

        predictions = pred_classes.clone()

        reject_mask = max_probs < THRESHOLD

        # -1 = unknown prediction
        predictions[reject_mask] = -1

        # ---------------------------------------------
        # Separate known and unknown GT samples
        # ---------------------------------------------

        known_mask = torch.isin(
            labels,
            train_classes_tensor
        )

        unknown_mask = ~known_mask

        # ---------------------------------------------
        # Known samples
        # ---------------------------------------------

        if known_mask.any():

            known_predictions = predictions[known_mask]
            known_labels = labels[known_mask]

            # Overall known statistics
            known_correct += (
                known_predictions == known_labels
            ).sum().item()

            known_total += known_labels.size(0)

            # Per-class statistics
            for class_id in train_classes:

                class_mask = known_labels == class_id

                if class_mask.any():

                    class_total[class_id] += (
                        class_mask.sum().item()
                    )

                    class_correct[class_id] += (
                        (
                            known_predictions[class_mask]
                            == class_id
                        )
                        .sum()
                        .item()
                    )

        # ---------------------------------------------
        # Unknown samples
        # ---------------------------------------------

        if unknown_mask.any():

            unknown_predictions = predictions[unknown_mask]

            unknown_total += unknown_mask.sum().item()

            # Correct if rejected
            unknown_correct += (
                unknown_predictions == -1
            ).sum().item()


# ==========================================================
# Results
# ==========================================================

# Per-class accuracy
per_class_accuracy = {}

for class_id in train_classes:

    if class_total[class_id] > 0:
        per_class_accuracy[class_id] = (
            class_correct[class_id]
            / class_total[class_id]
        )
    else:
        per_class_accuracy[class_id] = 0.0


# ----------------------------------------------------------
# Sample-weighted accuracy across ALL known samples
# ----------------------------------------------------------

overall_known_accuracy = (
    known_correct / known_total
    if known_total > 0
    else 0.0
)


# ----------------------------------------------------------
# Macro-average over known classes
# ----------------------------------------------------------

mean_known_accuracy = (
    sum(per_class_accuracy.values())
    / len(per_class_accuracy)
)


# ----------------------------------------------------------
# Unknown accuracy = unknown rejection rate
# ----------------------------------------------------------

unknown_accuracy = (
    unknown_correct / unknown_total
    if unknown_total > 0
    else 0.0
)


# ==========================================================
# Print results
# ==========================================================

print("Validation results")
print("------------------")

print(f"Known samples:   {known_total}")
print(f"Unknown samples: {unknown_total}")

print("\nPer-class accuracy:")

for class_id, accuracy in per_class_accuracy.items():

    print(
        f"  Class {class_id}: {accuracy:.4f} "
        f"({class_correct[class_id]}/{class_total[class_id]})"
    )

print(
    f"\nOverall known accuracy "
    f"(sample-weighted): {overall_known_accuracy:.4f} "
    f"({known_correct}/{known_total})"
)

print(
    f"Mean known-class accuracy "
    f"(macro): {mean_known_accuracy:.4f}"
)

print(
    f"Unknown accuracy: "
    f"{unknown_accuracy:.4f} "
    f"({unknown_correct}/{unknown_total})"
)

Validation results
------------------
Known samples:   352
Unknown samples: 34

Per-class accuracy:
  Class 0: 0.6667 (2/3)
  Class 1: 0.6667 (2/3)
  Class 2: 1.0000 (3/3)
  Class 3: 0.6667 (2/3)
  Class 4: 1.0000 (3/3)
  Class 5: 0.6667 (2/3)
  Class 6: 1.0000 (3/3)
  Class 7: 1.0000 (3/3)
  Class 8: 1.0000 (3/3)
  Class 9: 1.0000 (3/3)
  Class 10: 1.0000 (3/3)
  Class 11: 1.0000 (3/3)
  Class 12: 0.6667 (2/3)
  Class 13: 0.6667 (2/3)
  Class 14: 0.6667 (2/3)
  Class 15: 0.6667 (2/3)
  Class 16: 1.0000 (3/3)
  Class 17: 1.0000 (3/3)
  Class 18: 1.0000 (3/3)
  Class 19: 0.3333 (1/3)
  Class 20: 0.6667 (2/3)
  Class 21: 0.6667 (2/3)
  Class 22: 1.0000 (3/3)
  Class 23: 0.6667 (2/3)
  Class 24: 1.0000 (3/3)
  Class 25: 1.0000 (3/3)
  Class 26: 1.0000 (3/3)
  Class 27: 1.0000 (17/17)
  Class 28: 1.0000 (11/11)
  Class 29: 1.0000 (17/17)
  Class 30: 1.0000 (16/16)
  Class 31: 0.4375 (7/16)
  Class 32: 0.0000 (0/2)
  Class 33: 0.8824 (15/17)
  Class 34: 1.0000 (11/11)
  Class 35: 1.0000 (11

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_EPOCHS = 15
LR = 1e-3
THRESHOLD = 0.6


# -------------------------------------------------------
# DINOv2 model
# -------------------------------------------------------

dino = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

dino = dino.to(DEVICE)
dino.eval()

# Freeze DINO
for p in dino.parameters():
    p.requires_grad = False


# -------------------------------------------------------
# Class mapping
# -------------------------------------------------------

# Example:
# train folders: 1,2,3
# train_classes = [1,2,3]

train_classes = sorted(
    train_dataset.class_to_idx.values()
)

# Actual class id -> classifier output index
class_to_model_idx = {
    c: i for i, c in enumerate(train_classes)
}

# Tensor version for evaluation
train_classes_tensor = torch.tensor(
    train_classes,
    device=DEVICE
)


# -------------------------------------------------------
# Classifier head
# -------------------------------------------------------

classifier = nn.Linear(
    384,                         # DINOv2 ViT-S/14 feature dim
    len(train_classes)
).to(DEVICE)


optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=LR
)

criterion = nn.CrossEntropyLoss()


# -------------------------------------------------------
# Training
# -------------------------------------------------------

for epoch in range(NUM_EPOCHS):

    classifier.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # Convert dataset labels -> classifier indices
        labels = torch.tensor(
            [
                class_to_model_idx[x.item()]
                for x in labels
            ],
            device=DEVICE
        )

        with torch.no_grad():
            features = dino(images)

        logits = classifier(features)

        loss = criterion(
            logits,
            labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} "
        f"Loss: {running_loss/total:.4f} "
        f"Acc: {correct/total:.4f}"
    )


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 395MB/s]


Epoch 1/15 Loss: 1.4814 Acc: 0.6707
Epoch 2/15 Loss: 0.3388 Acc: 0.9087
Epoch 3/15 Loss: 0.1710 Acc: 0.9567
Epoch 4/15 Loss: 0.0945 Acc: 0.9760
Epoch 5/15 Loss: 0.0631 Acc: 0.9832


KeyboardInterrupt: 

In [63]:
# -------------------------------------------------------
# Validation with unknown rejection
# -------------------------------------------------------

classifier.eval()

known_correct = 0
known_total = 0

unknown_correct = 0
unknown_total = 0


with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        features = dino(images)

        logits = classifier(features)

        probs = F.softmax(
            logits,
            dim=1
        )

        max_probs, pred_idx = torch.max(
            probs,
            dim=1
        )


        # Convert classifier index back to real class id
        pred_classes = torch.tensor(
            [
                train_classes[i]
                for i in pred_idx.cpu().numpy()
            ],
            device=DEVICE
        )


        # Abstain if confidence is low
        predictions = pred_classes.clone()

        predictions[max_probs < THRESHOLD] = -1


        # Identify known/unknown GT labels
        known_mask = torch.isin(
            labels,
            train_classes_tensor
        )

        unknown_mask = ~known_mask


        # -------------------------
        # Known accuracy
        # -------------------------

        if known_mask.any():

            known_correct += (
                predictions[known_mask]
                ==
                labels[known_mask]
            ).sum().item()

            known_total += (
                known_mask.sum().item()
            )


        # -------------------------
        # Unknown accuracy
        # (correct rejection)
        # -------------------------

        if unknown_mask.any():

            unknown_correct += (
                predictions[unknown_mask]
                ==
                -1
            ).sum().item()

            unknown_total += (
                unknown_mask.sum().item()
            )


print("\nValidation")
print("----------------")

print(
    f"Known samples:   {known_total}"
)

print(
    f"Unknown samples: {unknown_total}"
)

if known_total:
    print(
        f"Known accuracy:  "
        f"{known_correct/known_total:.4f}"
    )

if unknown_total:
    print(
        f"Unknown accuracy:"
        f" {unknown_correct/unknown_total:.4f}"
    )


Validation
----------------
Known samples:   352
Unknown samples: 34
Known accuracy:  0.8182
Unknown accuracy: 0.5294


In [64]:
torch.save(
    classifier.state_dict(),
    "/content/drive/MyDrive/Colab Notebooks/dinov2_classifier.pth"
)

In [87]:
import os
import json
import torch
from PIL import Image
from torchvision import transforms


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(DEVICE)
model.eval()


# ------------------------------------------------------------
# Class mapping
# ------------------------------------------------------------

# Same mapping used during training
class_to_model_idx = train_dataset.class_to_idx
print(class_to_model_idx)
# ------------------------------------------------------------
# Preprocessing
# ------------------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # transforms.Normalize(
    #     mean=(0.485, 0.456, 0.406),
    #     std=(0.229, 0.224, 0.225)
    # )
])


# ------------------------------------------------------------
# Load annotations
# ------------------------------------------------------------

with open("/content/drive/MyDrive/Colab Notebooks/gondola_eval/boxes.json", "r") as f:
    boxes = json.load(f)

print(f"Found {len(boxes)} annotations")


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

correct = 0
total = 0

class_correct = {}
class_total = {}


with torch.no_grad():

    for i, annotation in enumerate(boxes):

        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        image_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/gondola_eval/", annotation["frame"])

        image = Image.open(image_path).convert("RGB")


        # ----------------------------------------------------
        # Crop bounding box
        # ----------------------------------------------------

        x1, y1, x2, y2 = annotation["bbox_xyxy"]

        crop = image.crop(
            (x1, y1, x2, y2)
        )


        # ----------------------------------------------------
        # Ground-truth class
        # ----------------------------------------------------

        gt_class = annotation["product_id"]

        if str(gt_class) not in class_to_model_idx:
          continue
        # Convert actual class ID -> model index
        gt_label = class_to_model_idx[str(gt_class)]


        # ----------------------------------------------------
        # Preprocess
        # ----------------------------------------------------

        input_tensor = transform(crop)

        input_tensor = input_tensor.unsqueeze(0)
        input_tensor = input_tensor.to(DEVICE)


        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        outputs = model(input_tensor)

        prediction = outputs.argmax(
            dim=1
        ).item()


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total += 1

        if prediction == gt_label:
            correct += 1
            is_correct = True
        else:
            is_correct = False


        # Per-class statistics
        if gt_class not in class_total:

            class_total[gt_class] = 0
            class_correct[gt_class] = 0

        class_total[gt_class] += 1

        if is_correct:
            class_correct[gt_class] += 1


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

overall_accuracy = correct / total

print("\n==============================")
print("Evaluation results")
print("==============================")

print(f"Total samples:    {total}")
print(f"Correct:          {correct}")
print(f"Overall accuracy: {overall_accuracy:.4f}")


print("\nPer-class accuracy:")

per_class_accuracy = {}

for class_id in sorted(class_total):

    accuracy = (
        class_correct[class_id]
        / class_total[class_id]
    )

    per_class_accuracy[class_id] = accuracy

    print(
        f"Class {class_id}: "
        f"{accuracy:.4f} "
        f"({class_correct[class_id]}/{class_total[class_id]})"
    )

{'10043': 0, '10053': 1, '10061': 2, '10062': 3, '10075': 4, '10086': 5, '10119': 6, '10147': 7, '10159': 8, '10160': 9, '10190': 10, '10226': 11, '10258': 12, '10325': 13, '10377': 14, '10408': 15, '10444': 16, '10445': 17, '10465': 18, '10470': 19, '10472': 20, '10489': 21, '10492': 22, '10495': 23, '10518': 24, '10519': 25, '12390': 26, '12406': 27, '12407': 28, '12408': 29, '12409': 30, '12410': 31, '12411': 32, '12412': 33, '12413': 34, '12414': 35, '12415': 36, '12416': 37, '12417': 38, '12418': 39, '12419': 40, '12420': 41, '12421': 42, '12425': 43, '12427': 44, '12675': 45, '12676': 46, '12677': 47, '12681': 48, '12683': 49, '12684': 50, '12685': 51, '12687': 52, '12688': 53, '12689': 54, '12690': 55, '12693': 56, '1293': 57, '251': 58, '541': 59}
Found 142 annotations
Processed 100/142 | Accuracy: 0.9400

Evaluation results
Total samples:    142
Correct:          134
Overall accuracy: 0.9437

Per-class accuracy:
Class 541: 1.0000 (3/3)
Class 1293: 0.6667 (2/3)
Class 12406: 1.0

In [89]:
# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

correct = 0
total = 0

camera_correct = {}
camera_total = {}

with torch.no_grad():

    for i, annotation in enumerate(boxes):

        # ----------------------------------------------------
        # Camera ID extraction
        # ----------------------------------------------------

        frame = annotation["frame"]

        # Assuming format: cameraXXX/filename.jpg
        camera_id = annotation["camera"]

        if camera_id not in camera_total:
            camera_total[camera_id] = 0
            camera_correct[camera_id] = 0


        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        image_path = os.path.join(
            "/content/drive/MyDrive/Colab Notebooks/gondola_eval/",
            frame
        )

        image = Image.open(image_path).convert("RGB")


        # ----------------------------------------------------
        # Crop bounding box
        # ----------------------------------------------------

        x1, y1, x2, y2 = annotation["bbox_xyxy"]

        crop = image.crop(
            (x1, y1, x2, y2)
        )


        # ----------------------------------------------------
        # Ground-truth class
        # ----------------------------------------------------

        gt_class = annotation["product_id"]

        if str(gt_class) not in class_to_model_idx:
            continue

        gt_label = class_to_model_idx[str(gt_class)]


        # ----------------------------------------------------
        # Preprocess
        # ----------------------------------------------------

        input_tensor = transform(crop)
        input_tensor = input_tensor.unsqueeze(0).to(DEVICE)


        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        outputs = model(input_tensor)

        prediction = outputs.argmax(dim=1).item()


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total += 1
        camera_total[camera_id] += 1

        if prediction == gt_label:
            correct += 1
            camera_correct[camera_id] += 1


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

overall_accuracy = correct / total

print("\n==============================")
print("Overall evaluation")
print("==============================")

print(f"Total samples:    {total}")
print(f"Correct:          {correct}")
print(f"Accuracy:         {overall_accuracy:.4f}")


print("\n==============================")
print("Accuracy per camera")
print("==============================")

for camera_id in sorted(camera_total):

    acc = (
        camera_correct[camera_id]
        / camera_total[camera_id]
    )

    print(
        f"{camera_id}: "
        f"{acc:.4f} "
        f"({camera_correct[camera_id]}/{camera_total[camera_id]})"
    )


Overall evaluation
Total samples:    142
Correct:          134
Accuracy:         0.9437

Accuracy per camera
cam689: 0.9583 (46/48)
cam748: 0.9348 (43/46)
cam749: 0.9375 (45/48)
